In [2]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os

C:\Users\chimu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
ollama_api_key = os.getenv("OLLAMA_API_KEY")


In [4]:
model=ChatOllama(
    model="gemma4",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {"Authorization": f"Bearer {ollama_api_key}"}
    }
)

In [5]:
#create state
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [6]:
#declare constants
CREATE_OUTLINE="create_outline"
CREATE_BLOG="create_blog"

In [ ]:
def create_outline(state:BlogState)->BlogState:
    #extract title from state
    title=state["title"]
    
    #form prompt
    prompt=f"Write an detailed outline for a blog on a topic {title}"
    
    #generate outline
    outline=model.invoke(prompt).content
    
    #update state
    state["outline"]=outline
    
    return state

In [ ]:
def create_blog(state:BlogState)->BlogState:
    #extract outline from state
    title=state["title"]
    outline=state["outline"]
    
    #form prompt
    prompt=f"Write a detailed blog on a topic {title} using following outline\n {outline}"
    
    #generate blog content
    blog_content=model.invoke(prompt).content
    
    #update state
    state["content"]=blog_content
    
    return state

In [9]:
graph=StateGraph(BlogState)
# add nodes to graph
graph.add_node(CREATE_OUTLINE,create_outline)
graph.add_node(CREATE_BLOG,create_blog)

#add edges to graph
graph.add_edge(START,CREATE_OUTLINE)
graph.add_edge(CREATE_OUTLINE,CREATE_BLOG)
graph.add_edge(CREATE_BLOG,END)

In [10]:
# compile graph
workflow=graph.compile()

In [11]:
#initial state
initial_state={"title":"Rise of AI in India"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in India', 'outline': 'This is a comprehensive outline for a long-form, authoritative blog post titled **"The Digital Renaissance: The Rise of AI in India."** \n\nDepending on your target audience (business leaders, tech enthusiasts, or the general public), you can adjust the tone from "technical" to "accessible."\n\n---\n\n# Blog Title Options:\n*   *The Digital Renaissance: The Rise of AI in India*\n*   *From Back-Office to Brain-Trust: How AI is Transforming India*\n*   *AI in India: Challenges, Opportunities, and the Road to 2030*\n\n## I. Introduction\n*   **The Hook:** A compelling statistic about India’s internet penetration or the growth of AI startups in the last 3 years.\n*   **The Context:** Briefly transition from India as the "World’s Outsourcing Hub" (IT services) to India as an "AI Innovation Hub."\n*   **Thesis Statement:** AI is not just a tool for efficiency in India; it is a catalyst for solving systemic socio-economic challenges at scale.\n*   

In [12]:
print(final_state["content"])

# The Digital Renaissance: The Rise of AI in India

With over 800 million internet users and a startup ecosystem that is one of the largest in the world, India is no longer just "connecting" to the digital world—it is beginning to architect it. In the last three years, the proliferation of Generative AI has acted as a catalyst, sparking a surge in AI-first startups and a fundamental shift in how the nation approaches productivity.

For decades, India was known as the "World’s Outsourcing Hub," the reliable back-office providing IT services and BPO support to the West. But a paradigm shift is underway. We are witnessing a transition from **service-based delivery to product-based innovation.** 

AI in India is not merely a tool for corporate efficiency; it is becoming a catalyst for solving systemic socio-economic challenges at a scale unseen anywhere else on earth. From the remote farms of Vidarbha to the tech parks of Bengaluru, the "Digital Renaissance" has arrived.

---

## The Found